# RUHSOLD baseline and cross-script alignment pilot

Group 18 ? CS 5316 NLP. This notebook reproduces the Task 1 supervised baseline and provides a clearly labeled scaffold for the proposed prompting novelty. It does not claim LLM results. Run from the project root.

## 1. Setup and data audit

Task 1 labels are `0=abusive/offensive`, `1=normal` (per the dataset label definitions). The provided Task 1 train/validation/test splits are used as supplied. Task 2 validation is excluded because it duplicates training.

In [ ]:
from pathlib import Path
import csv, json, time
from collections import Counter
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

ROOT = Path.cwd()
DATA = ROOT / 'data/2020paper/roman_urdu_hate_speech-main'
OUT = ROOT / 'results'; OUT.mkdir(exist_ok=True)
LABEL_NAMES = {0: 'abusive/offensive', 1: 'normal'}
def load_split(name):
    path = DATA / f'task_1_{name}.tsv'
    with path.open(encoding='utf-8-sig', newline='', errors='replace') as f:
        rows = list(csv.reader(f, delimiter='\t'))
    rows = [r for r in rows if len(r)==2 and r[0].strip() and r[1].strip() in {'0','1'}]
    return [r[0] for r in rows], [int(r[1]) for r in rows]
X_train,y_train=load_split('train'); X_val,y_val=load_split('validation'); X_test,y_test=load_split('test')
print('sizes:',len(X_train),len(X_val),len(X_test))
print('train labels:',Counter(y_train),'test labels:',Counter(y_test))

## 2. Character n-gram TF-IDF + linear SVM

The baseline uses character n-grams (2?5) and LinearSVC with C=1.0. We preserve original dataset label IDs for computation; names follow the source label definitions.

In [ ]:
model = Pipeline([('tfidf', TfidfVectorizer(analyzer='char', ngram_range=(2,5))), ('svm', LinearSVC(C=1.0))])
t0=time.perf_counter(); model.fit(X_train,y_train); train_seconds=time.perf_counter()-t0
pred=model.predict(X_test)
print(classification_report(y_test,pred,labels=[0,1],target_names=[LABEL_NAMES[0],LABEL_NAMES[1]],digits=4,zero_division=0))
print('Confusion matrix [abusive/offensive, normal]:\n',confusion_matrix(y_test,pred,labels=[0,1]))
print('Accuracy:',accuracy_score(y_test,pred),'train seconds:',round(train_seconds,3))

In [ ]:
report=classification_report(y_test,pred,labels=[0,1],target_names=[LABEL_NAMES[0],LABEL_NAMES[1]],output_dict=True,zero_division=0)
metrics={'dataset':'RUHSOLD Task 1 coarse-grained','train_n':len(X_train),'validation_n':len(X_val),'test_n':len(X_test),'model':'char TF-IDF (2-5), LinearSVC(C=1.0)','accuracy':accuracy_score(y_test,pred),'macro_f1':report['macro avg']['f1-score'],'weighted_f1':report['weighted avg']['f1-score'],'class_report':report,'confusion_matrix':confusion_matrix(y_test,pred,labels=[0,1]).tolist(),'label_mapping':LABEL_NAMES}
(OUT/'ruhsold_task1_svm_metrics.json').write_text(json.dumps(metrics,indent=2),encoding='utf-8')
pd.DataFrame({'text':X_test,'gold_label':y_test,'gold_name':[LABEL_NAMES[y] for y in y_test],'prediction':pred,'prediction_name':[LABEL_NAMES[int(y)] for y in pred]}).to_csv(OUT/'ruhsold_task1_svm_predictions.csv',index=False,encoding='utf-8-sig')
metrics

## 3. Proposed novelty: cross-script alignment prompting

The project proposal injects Roman Urdu ? Urdu-script lexical alignment examples into a prompt. Conditions planned: zero-shot, standard few-shot, and alignment prompting with N=5/15 mappings. The cells below prepare an auditable prompt and expose the expected API configuration; no key is embedded and no inference is performed here. Before reporting novelty results, freeze the dictionary and examples, run the same held-out test subset across conditions, log model/version, prompt length, latency, failures, and macro-F1.

In [ ]:
ALIGNMENT_EXAMPLES = [
    ('nahi', '????'), ('acha', '????'), ('kyun', '????'),
    ('mujhe', '????'), ('tum', '??'), ('hai', '??'),
    ('kya', '???'), ('log', '???'), ('bohat', '???'),
    ('mera', '????'), ('ap', '??'), ('dost', '????'),
    ('ghar', '???'), ('baat', '???'), ('karna', '????')]
def alignment_prompt(comment, n=5):
    if n not in (0,5,15): raise ValueError('n must be 0, 5, or 15')
    mapping='\n'.join(f'{roman} -> {urdu}' for roman,urdu in ALIGNMENT_EXAMPLES[:n])
    return ('Classify this Roman Urdu comment as Abusive or Not abusive.\n'
            + (f'Roman Urdu to Urdu-script examples:\n{mapping}\n' if n else '')
            + f'Comment: {comment}\nAnswer with one label only: Abusive or Not abusive.')
print(alignment_prompt('tum kya kar rahe ho',n=5))

## 4. Interpretation and limitations

The supervised baseline establishes a reproducible reference point. The held-out test result is not an LLM comparison. The validation split is reserved and currently unused. The primary report should distinguish the available 2020 RUHSOLD release from the separate 2026 anchor paper?s experimental setup; exact replication requires matching that paper?s data, models, prompts, and evaluation protocol. The proposed extension remains an implementation plan until API/local-model runs are performed.